# 🚀 PROJECT BLUEPRINT: AGENTIC STOCK FORECASTING SYSTEM (VINGROUP ECOSYSTEM) 

## 📌 Tổng quan Hệ thống (System Overview)

Một hệ thống Pipeline hoàn toàn tự động (end-to-end), chịu trách nhiệm thu thập dữ liệu chuỗi thời gian của rổ cổ phiếu họ Vingroup (VN30), tạo các dự báo có kèm khoảng tin cậy (80% & 95%), và triển khai một đội ngũ AI Agent (Agentic Workflow) để tự động đánh giá, diễn giải ngữ cảnh thị trường, và tự động tinh chỉnh mô hình.

**Kiến trúc Cốt lõi:**
`[Data Ingestion]` $\rightarrow$ `[Processing & Engineering]` $\rightarrow$ `[LightGBM Quantile & Holdout Eval]` $\rightarrow$ `[LangGraph Evaluator]` $\rightarrow$ `[Reporting & Retraining]`

---

## 📥 BƯỚC 1: LIVE DATA INGESTION (Thu thập dữ liệu thời gian thực)

Hệ thống kết nối với các nguồn API công khai để tự động kéo dữ liệu thị trường hàng ngày.

*   **Công cụ:** Thư viện `vnstock` (Python).
*   **Danh mục mục tiêu:** Cổ phiếu **VHM, VIC, VPL, VRE**.
*   **Dữ liệu bối cảnh thị trường (Context):** Chỉ số VN30, VN30 Futures (Phái sinh).
*   **Khung thời gian (Time Interval):** Dữ liệu theo ngày (1D).

## ⚙️ BƯỚC 2: DATA PROCESSING & FEATURE ENGINEERING (Xử lý & Khai phá Đặc trưng)

Biến đổi dữ liệu thô thành bộ tính năng (features) giàu thông tin phục vụ cho mô hình học máy.

*   **Làm sạch dữ liệu (Data Cleaning):** Loại bỏ giá trị `NaN`, xử lý dữ liệu trùng lặp (duplicates), nội suy để điền các khoảng trống (fill gap) và xử lý ngoại lai (outliers).
*   **Lưu trữ (Storage):** Sử dụng cơ sở dữ liệu nhẹ `SQLite` (thông qua `SQLAlchemy`) cho dữ liệu cấu trúc và backup qua file `CSV`.
*   **Feature Engineering (Đặc trưng kỹ thuật):**
    *   *Giá & Khối lượng:* Lợi nhuận (Return), % thay đổi giá, % thay đổi khối lượng, Đường trung bình động (Rolling Avg), Momentum.
    *   *Độ biến động:* Độ lệch chuẩn trượt (Rolling std), Biến động lịch sử (Historical volatility).
    *   *Độ trễ (Lag features):* Độ trễ chu kỳ $1, 3, 7, 14$ ngày.

## 📈 BƯỚC 3: MODELING & FORECASTING (Mô hình hóa & Đánh giá Holdout)

Tạo dự báo 7 ngày tiếp theo đáp ứng trọn vẹn yêu cầu về phân phối xác suất và báo cáo lỗi mô hình.

*   **Mô hình (Model):** `LightGBM` kết hợp với **Quantile Regression**. Mô hình sẽ huấn luyện đồng thời trên 5 phân vị mục tiêu: $q \in \{0.025, 0.1, 0.5, 0.9, 0.975\}$.
*   **Đầu ra Dự báo (Output):**
    *   *Point forecast:* Giá trị tại phân vị $q = 0.5$ (Median).
    *   *Confidence Interval 80%:* Biên độ từ $q = 0.1$ đến $q = 0.9$.
    *   *Confidence Interval 95%:* Biên độ từ $q = 0.025$ đến $q = 0.975$.
*   **Đánh giá Holdout (Holdout Evaluation):** Tách 30 ngày cuối cùng của tập dữ liệu lịch sử làm tập Holdout. Tính toán các chỉ số bắt buộc **MAE, RMSE, MAPE** trên tập này trước khi đẩy toàn bộ dự báo sang cho AI Agent.

## 🤖 BƯỚC 4: AI AGENT EVALUATION (Tác nhân AI Đánh giá - Core Logic)

Phân hệ cốt lõi sử dụng thiết kế luồng đồ thị `LangGraph` (với `Google Gemini API` hoặc LLM tương đương) để tự động kiểm soát chất lượng dự báo và ra quyết định.

*   **[NODE 1]: VALIDATE THE PREDICT INTERVAL**
    *   Kiểm tra lỗi Quantile Crossing (khoảng tin cậy bị chéo/ngược). Sử dụng rules để ghim (clip/fix) lại, kích hoạt cảnh báo (Warning) cho Node sau.
*   **[NODE 2]: EVALUATE THE MODEL OUTPUT (Đánh giá thực tế)**
    *   *Tại thời điểm $T = 1$:* Đánh giá tính hợp lý (Plausibility) so với lịch sử.
    *   *Tại thời điểm $T > 1$ (Đã có Ground Truth):* Tính toán accuracy/error hiện tại khi có dữ liệu mới.
    *   *Quyết định:* Nếu **PASS** $\rightarrow$ Đi thẳng `[NODE 3]`. Nếu **ABNORMAL** $\rightarrow$ Đi đến `[NODE 2.1]`.
*   **[NODE 2.1]: CONTEXTUALIZE & SENTIMENT ANALYSIS (Tìm nguyên nhân)**
    *   Agent tạo câu lệnh truy vấn tối ưu $\rightarrow$ Dùng MCP/Tool kéo tin tức từ Google News/Vietstock RSS $\rightarrow$ Thực hiện Sentiment Analysis.
    *   *Quyết định:* Nếu tin tức giải thích được sự bất thường (VD: Bắt bớ, bùng nổ lợi nhuận) $\rightarrow$ Ghi nhận Warning, giữ nguyên Model, đi đến `[NODE 3]`. Nếu không giải thích được $\rightarrow$ Chuyển sang `[NODE 2.2]`.
*   **[NODE 2.2]: AGENTIC IMPROVEMENT & RETRAIN (Tinh chỉnh chủ động)**
    *   Agent phân tích sâu các chỉ số Diagnose (Overfitting/Underfitting) và **chủ động ra quyết định điều chỉnh cấu hình (on the fly):**
        *   *Kịch bản 1:* Nếu underfitting $\rightarrow$ Agent gọi tool thay đổi file config: tăng `max_depth` hoặc giảm `learning_rate` $\rightarrow$ Retrain.
        *   *Kịch bản 2:* Nếu thị trường vừa có cú shock cấu trúc $\rightarrow$ Agent gọi tool tự động tăng trọng số (sample weights) cho 7 ngày dữ liệu gần nhất $\rightarrow$ Retrain.
    *   Quá trình này lặp lại tối đa $N$ lần trong ngày. Vượt quá sẽ văng lỗi nghiêm trọng gửi về `[NODE 3]`.
*   **[NODE 3]: GENERATE RECOMMENDATION**
    *   Tổng hợp Metrics (từ Bước 3 và Bước 4), Tin tức (Node 2.1), và Trạng thái Tinh chỉnh (Node 2.2) để đưa ra khuyến nghị cuối cùng: Trust (Tin tưởng), Caution (Thận trọng), hay Manual Intervention (Cần con người can thiệp).

## 📋 BƯỚC 5: REPORTING (Hệ thống Báo cáo Đa định dạng)

*   **Báo cáo JSON:** Chứa toàn bộ raw data, array dự báo (point + 80% + 95%), errors (MAE, RMSE, MAPE) và logs quyết định của Agent (Đáp ứng chuẩn System-readable).
*   **Báo cáo Markdown:** Báo cáo tóm tắt hàng ngày cho con người đọc, giải thích tại sao Agent đưa ra quyết định retrain hay giữ nguyên model.
*   **Giao diện HTML (Mở rộng):** Sinh tự động file HTML trực quan hóa biểu đồ nến (candlestick chart) kèm các dải khoảng tin cậy.


```text
AGENTIC_STOCK_FORECASTING_SYSTEM/
│
├── configs/                  # Chứa các file cấu hình (thay đổi on-the-fly)
│   ├── model_config.yaml     # Params cho LightGBM
│   └── agent_config.yaml     # Cấu hình prompt, thresholds cho Agent
│
├── data/                     # Thư mục chứa dữ liệu (KHÔNG push lên Git)
│   ├── raw/                  # Dữ liệu thô tải từ vnstock
│   ├── processed/            # Dữ liệu đã qua Feature Engineering
│   └── database.sqlite       # Cơ sở dữ liệu SQLite cục bộ
│
├── logs/                     # File log chạy hàng ngày của hệ thống
│
├── models/                   # Lưu trữ model LightGBM (.pkl hoặc .txt)
│
├── reports/                  # Thư mục xuất báo cáo hàng ngày (Component 5)
│   ├── json/
│   ├── markdown/
│   └── html/
│
├── src/                      # Source code chính của hệ thống
│   ├── __init__.py
│   ├── ingestion/            # BƯỚC 1: Lấy dữ liệu
│   │   ├── __init__.py
│   │   ├── vnstock_api.py    # Code gọi API lấy giá cổ phiếu
│   │   └── context_api.py    # Code lấy VN30, VN30F
│   │
│   ├── processing/           # BƯỚC 2: Xử lý & DB
│   │   ├── __init__.py
│   │   ├── cleaner.py        # Xử lý NaN, Outliers
│   │   ├── features.py       # Feature Engineering (Lag, Volatility...)
│   │   └── db_manager.py     # SQLAlchemy lưu SQLite
│   │
│   ├── modeling/             # BƯỚC 3: LightGBM
│   │   ├── __init__.py
│   │   ├── trainer.py        # Train model (có Holdout eval)
│   │   └── predictor.py      # Tạo dự báo 5 phân vị (Quantile)
│   │
│   ├── agent/                # BƯỚC 4: LangGraph Evaluator
│   │   ├── __init__.py
│   │   ├── graph.py          # Định nghĩa các Node và luồng Graph
│   │   ├── nodes.py          # Logic chi tiết của từng Node
│   │   ├── state.py          # Định nghĩa State (cấu trúc dữ liệu truyền giữa các Node)
│   │   └── tools.py          # MCP, Search News, Tool sửa config
│   │
│   └── reporting/            # BƯỚC 5: Xuất Report
│       ├── __init__.py
│       └── generator.py      # Sinh JSON, MD, HTML (Plotly)
│
├── utils/                    # Các hàm tiện ích dùng chung
│   ├── __init__.py
│   ├── logger.py             # Cấu hình logging
│   └── helpers.py
│
├── notebooks/                # Chứa file .ipynb để test nháp code trước khi đưa vào src/
│
├── .env                      # Chứa API Key (Gemini API, v.v...)
├── .gitignore                # Bỏ qua data/, logs/, models/, .env
├── requirements.txt          # Danh sách thư viện
└── main.py                   # ĐIỂM CHẠY CHÍNH CỦA PIPELINE (Daily trigger)
```

# Codebase Review

Project name: **Agentic Vingroup Quant Forecasting System**

This document reviews the current repository as implemented. It is intentionally descriptive: it explains what the code does today and does not propose or apply any code changes.

## 1. Current Project Structure

```text
.
|-- main.py
|-- README.md
|-- requirements.txt
|-- configs/
|   |-- agent_config.yaml
|   `-- model_config.yaml
|-- data/
|   |-- database.sqlite
|   |-- processed/
|   `-- raw/
|-- docs/
|   `-- CODEBASE_REVIEW.md
|-- logs/
|   `-- YYYY-MM-DD_system.log
|-- models/
|-- notebooks/
|   |-- blueprint.ipynb
|   `-- workflow.ipynb
|-- reports/
|   |-- html/
|   |-- json/
|   `-- markdown/
|-- src/
|   |-- agent/
|   |-- ingestion/
|   |-- modeling/
|   |-- monitoring/
|   |-- processing/
|   |-- reporting/
|   `-- risk/
|-- test_agent.py
|-- test_ingestion.py
|-- test_modeling.py
|-- test_monitoring_risk.py
|-- test_processing.py
|-- test_validation.py
`-- utils/
```

## 2. Folder Roles

| Folder | Role |
|---|---|
| `configs/` | YAML configuration for model parameters, walk-forward validation, agent thresholds, and prompt placeholders. |
| `data/` | Local data storage. The current implementation writes/reads the main data through `data/database.sqlite`. |
| `docs/` | Human documentation for reviewers and maintainers. |
| `logs/` | Runtime logs generated by `utils/logger.py`. |
| `models/` | Present in the repository, but the current code does not persist LightGBM model artifacts here. |
| `notebooks/` | Exploratory notebooks / blueprint material. They are not used by the runtime pipeline. |
| `reports/` | Generated JSON, Markdown, and HTML reports. |
| `src/agent/` | LangGraph workflow, Gemini calls, agent state schema, news tool, and hyperparameter adjustment tool. |
| `src/ingestion/` | vnstock ingestion for Vingroup stocks and market context symbols. |
| `src/modeling/` | LightGBM quantile model, 7-day forecast wrapper, holdout evaluation, and walk-forward validation. |
| `src/monitoring/` | Regime detection and drift detection. |
| `src/processing/` | Feature engineering, context feature joining, and SQLite storage helpers. |
| `src/reporting/` | JSON, Markdown, and HTML report generation. |
| `src/risk/` | Research-only risk engine and preliminary signal logic. |
| `utils/` | Logger and small helper functions. |

## 3. Important Files

| File | Role |
|---|---|
| `main.py` | End-to-end entry point. Runs ingestion, processing, modeling, monitoring, risk, LangGraph agent, and reports. |
| `src/ingestion/vnstock_api.py` | Fetches historical daily OHLCV using `vnstock`. |
| `src/processing/features.py` | Creates technical, lag, calendar, and context features. |
| `src/processing/cleaner.py` | Orchestrates feature generation and saves raw/processed tables to SQLite. |
| `src/processing/db_manager.py` | SQLite read/write helpers. |
| `src/modeling/trainer.py` | Defines `QuantileLightGBM`, prepares targets, runs holdout evaluation, and trains per-horizon quantile models. |
| `src/modeling/predictor.py` | Public forecast wrapper: `generate_7_day_forecast`. |
| `src/modeling/validation.py` | Walk-forward validation module and validation metric dataclasses. |
| `src/monitoring/regime_detector.py` | Detects volatility, trend, and liquidity regimes. |
| `src/monitoring/drift_detector.py` | Detects feature, target, and concept drift using simple explainable tests. |
| `src/risk/risk_engine.py` | Converts forecast quantiles and monitoring reports into risk statistics and a preliminary research signal. |
| `src/agent/state.py` | `AgentState` TypedDict schema. |
| `src/agent/graph.py` | LangGraph node graph and router logic. |
| `src/agent/nodes.py` | Agent node implementation, Gemini prompts, governance comparison, and final signal formatting. |
| `src/agent/tools.py` | Vietstock RSS news scan and model hyperparameter adjustment. |
| `src/reporting/generator.py` | Report generation for JSON, Markdown, and HTML. |
| `configs/model_config.yaml` | LightGBM and walk-forward settings. |
| `configs/agent_config.yaml` | Agent thresholds and prompt placeholders. |
| `utils/logger.py` | Console and file logger factory. |

## 4. Entry Point

The entry point is:

```text
main.py
```

When executed directly, it checks `GOOGLE_API_KEY` and runs:

```python
run_daily_pipeline(target_ticker="VIC")
```

## 5. Pipeline Flow

The pipeline runs in this order:

1. Load `.env` and initialize logging.
2. Generate `run_id`.
3. Fetch two years of data using `get_vingroup_and_context_data`.
4. Save raw data to SQLite.
5. Generate features for each Vingroup stock and join VN30/VN30F1M context features.
6. Save processed stock tables to SQLite.
7. Load the selected target table, e.g. `processed_VIC`.
8. Generate LightGBM quantile forecasts for horizons `T+1` to `T+7`.
9. Run holdout evaluation.
10. Run walk-forward validation.
11. Detect market regime.
12. Detect drift by splitting processed data into reference and recent windows.
13. Calculate risk report.
14. Invoke LangGraph agent.
15. If the model is abnormal, contextualize news and run retrain/governance loop.
16. Produce final research signal.
17. Generate JSON, Markdown, and HTML reports.

## 6. Main Functions and Classes by Module

### `main.py`

| Function | Purpose |
|---|---|
| `run_daily_pipeline(target_ticker="VIC")` | End-to-end orchestration. |
| `_split_reference_current(df, current_window=60)` | Splits processed data for drift detection. |

### `src/ingestion/vnstock_api.py`

| Function | Purpose |
|---|---|
| `fetch_historical_data(symbols, start_date, end_date, data_type="stock")` | Fetches historical OHLCV for a symbol list. |
| `get_vingroup_and_context_data(start_date, end_date)` | Fetches VIC, VHM, VRE, VPL, VN30, and VN30F1M. |

### `src/processing/features.py`

| Function | Purpose |
|---|---|
| `generate_technical_features(df)` | Creates stock-level features from OHLCV and date index. |
| `generate_context_features(context_df, prefix)` | Creates context return and close features for VN30/VN30F1M. |

### `src/processing/cleaner.py`

| Function | Purpose |
|---|---|
| `process_and_save_data(raw_data_dict)` | Runs feature engineering and writes raw/processed tables to SQLite. |

### `src/processing/db_manager.py`

| Function | Purpose |
|---|---|
| `get_engine()` | Creates SQLAlchemy SQLite engine. |
| `save_to_sqlite(df, table_name, replace=True)` | Writes DataFrame to SQLite. |
| `load_from_sqlite(table_name)` | Reads table from SQLite with `date` as index. |

### `src/modeling/trainer.py`

| Function/Class | Purpose |
|---|---|
| `load_config()` | Reads `configs/model_config.yaml`. |
| `QuantileLightGBM` | Main LightGBM quantile model wrapper. |
| `QuantileLightGBM.prepare_data(df, step=1)` | Creates future return target for a given horizon. |
| `QuantileLightGBM.evaluate_holdout(df)` | Runs one-step holdout evaluation over the last 30 rows. |
| `QuantileLightGBM.train_and_predict_step(df, X_last, step)` | Trains quantile models for a horizon and returns price quantile forecasts. |

### `src/modeling/predictor.py`

| Function | Purpose |
|---|---|
| `generate_7_day_forecast(df)` | Runs holdout, walk-forward validation, and direct multi-step quantile forecasts for 7 days. |

### `src/modeling/validation.py`

| Function/Class | Purpose |
|---|---|
| `WalkForwardConfig` | Dataclass for walk-forward parameters. |
| `ValidationMetrics` | Dataclass for aggregate validation metrics. |
| `FoldMetrics` | Dataclass for each validation fold. |
| `ValidationReport` | Dataclass for full validation report. |
| `load_walk_forward_config(...)` | Reads validation config from YAML. |
| `load_model_params(...)` | Reads LightGBM params from YAML. |
| `run_walk_forward_validation(...)` | Public dict-returning validation wrapper. |
| `walk_forward_validate(...)` | Main walk-forward validation implementation. |

### `src/monitoring/regime_detector.py`

| Function | Purpose |
|---|---|
| `detect_regime(processed_df)` | Detects volatility, trend, and liquidity regimes. |

### `src/monitoring/drift_detector.py`

| Function | Purpose |
|---|---|
| `detect_drift(reference_df, current_df, validation_metrics=None)` | Detects feature drift, target drift, and concept drift. |

### `src/risk/risk_engine.py`

| Function | Purpose |
|---|---|
| `calculate_risk_report(forecast_data, validation_metrics, regime_report=None, drift_report=None)` | Computes risk metrics and preliminary research signal. |

### `src/agent/graph.py`

| Function | Purpose |
|---|---|
| `build_agent_graph()` | Builds the LangGraph workflow. |
| `route_after_evaluate(state)` | Routes after model evaluation. |
| `route_after_contextualize(state)` | Routes after news context. |
| `route_after_improve(state)` | Routes after governance/retrain. |

### `src/agent/nodes.py`

| Function | Purpose |
|---|---|
| `node_validate(state)` | Fixes quantile crossing in forecast output if necessary. |
| `node_evaluate(state)` | Compares holdout MAPE against configured threshold. |
| `node_contextualize(state)` | Runs RSS scan and optionally Gemini evidence-based context classification. |
| `node_improve(state)` | Adjusts config, retrains challenger, compares governance gates, accepts or rolls back. |
| `node_recommend(state)` | Uses risk report as controlling signal and asks Gemini for committee-style summary. |
| `_compare_champion_challenger(...)` | Governance comparison using MAPE, directional accuracy, interval coverage, and RMSE. |

### `src/agent/tools.py`

| Function | Purpose |
|---|---|
| `tool_search_vietstock_news(ticker)` | Searches Vietstock RSS and returns structured news evidence. |
| `tool_adjust_model_hyperparams(adjustment_type)` | Mutates `configs/model_config.yaml` for retrain experiments. |

### `src/reporting/generator.py`

| Function | Purpose |
|---|---|
| `generate_reports(state)` | Writes JSON, Markdown, and HTML reports. |
| `ensure_directories()` | Ensures report folders exist. |

## 7. Data Flow Through the Pipeline

```text
vnstock daily OHLCV
    -> raw_data dict by symbol
    -> SQLite raw_* tables
    -> technical features per stock
    -> VN30/VN30F1M context join
    -> SQLite processed_* tables
    -> QuantileLightGBM prepare_data()
    -> transient future return target
    -> holdout + walk-forward validation
    -> 7-day quantile price forecast
    -> regime/drift/risk reports
    -> LangGraph AgentState
    -> final research signal
    -> JSON/Markdown/HTML reports
```

## 8. Agent Workflow

The graph has 5 nodes:

```text
node_validate
    -> node_evaluate
        -> node_recommend              if PASS
        -> node_contextualize          if ABNORMAL and retry_count == 0
        -> node_improve                if ABNORMAL and retry_count > 0
node_contextualize
    -> node_recommend                  if BLACK_SWAN or DATA_ISSUE
    -> node_improve                    otherwise
node_improve
    -> node_validate                   if retry_count < max_retries
    -> node_recommend                  if max retries reached
node_recommend
    -> END
```

Gemini is used in:

- `_invoke_context_committee`, only when news is found.
- `_invoke_recommendation_committee`, for final committee-style text.

The controlling research signal comes from `risk_report["preliminary_signal"]`, not from unbounded LLM speculation.

## 9. Report Output Locations

Reports are written by `src/reporting/generator.py`:

| Format | Path pattern | Purpose |
|---|---|---|
| JSON | `reports/json/{ticker}_report_{YYYY-MM-DD}.json` | Machine-readable full state and report metadata. |
| Markdown | `reports/markdown/{ticker}_report_{YYYY-MM-DD}.md` | Human-readable Quant Risk Committee report. |
| HTML | `reports/html/{ticker}_report_{YYYY-MM-DD}.html` | Browser-friendly report with Plotly fan chart. |

## 10. Config Controls

| Config | Fields | Used by |
|---|---|---|
| `configs/model_config.yaml` | `lightgbm_params`, `walk_forward_validation` | Model training, holdout, walk-forward validation, retrain experiments. |
| `configs/agent_config.yaml` | `thresholds.max_mape`, `thresholds.max_retries`, prompt placeholders | Agent evaluation and retrain loop routing. |

Current implementation note: the current working copy inspected here has `configs/agent_config.yaml` set to `max_mape: 0.03`. If the assessment scenario expects the demo threshold to be 1%, review this config manually before running the final assessment. This document does not change config values.

## Feature Inventory

Model usage rule: `QuantileLightGBM.prepare_data` uses every processed DataFrame column except `ticker`, `date`, and `target`. In the current processed SQLite tables, this results in 28 model input columns.

| Feature | Source | Logic | Module | Used by Model | Explanation |
|---|---|---|---|---|---|
| `open` | Raw vnstock OHLCV | Raw daily open price | `vnstock_api.py` / raw data | yes | Price at market open. |
| `high` | Raw vnstock OHLCV | Raw daily high price | `vnstock_api.py` / raw data | yes | Highest traded price of the day. |
| `low` | Raw vnstock OHLCV | Raw daily low price | `vnstock_api.py` / raw data | yes | Lowest traded price of the day. |
| `close` | Raw vnstock OHLCV | Raw daily close price | `vnstock_api.py` / raw data | yes | Last daily price; also target base column. |
| `volume` | Raw vnstock OHLCV | Raw daily volume | `vnstock_api.py` / raw data | yes | Trading activity level. |
| `ticker` | Raw vnstock OHLCV | Symbol identifier | `vnstock_api.py` | no | Non-numeric identifier excluded by model. |
| `daily_return` | `close` | `close.pct_change()` | `features.py` | yes | One-day percentage price change. |
| `vol_change` | `volume` | `volume.pct_change()` | `features.py` | yes | One-day percentage volume change. |
| `ma_7` | `close` | 7-day rolling mean | `features.py` | yes | Short-term moving average. |
| `ma_14` | `close` | 14-day rolling mean | `features.py` | yes | Medium-short moving average. |
| `volatility_7` | `daily_return` | 7-day rolling standard deviation | `features.py` | yes | Recent realized volatility. |
| `rsi_14` | `close` | `ta.momentum.RSIIndicator(..., window=14).rsi()` | `features.py` | yes | Momentum oscillator. |
| `macd` | `close` | `ta.trend.MACD(close).macd()` | `features.py` | yes | Trend/momentum indicator. |
| `atr_14` | `high`, `low`, `close` | 14-day Average True Range | `features.py` | yes | Range-based volatility indicator. |
| `roc_7` | `close` | 7-day Rate of Change | `features.py` | yes | Recent price momentum. |
| `day_of_week` | Date index | `df.index.dayofweek` | `features.py` | yes | Calendar day effect. |
| `month` | Date index | `df.index.month` | `features.py` | yes | Calendar month effect. |
| `close_lag_1` | `close` | `close.shift(1)` | `features.py` | yes | Yesterday close. |
| `return_lag_1` | `daily_return` | `daily_return.shift(1)` | `features.py` | yes | Yesterday return. |
| `close_lag_3` | `close` | `close.shift(3)` | `features.py` | yes | Close three trading days ago. |
| `return_lag_3` | `daily_return` | `daily_return.shift(3)` | `features.py` | yes | Return three trading days ago. |
| `close_lag_7` | `close` | `close.shift(7)` | `features.py` | yes | Close seven trading days ago. |
| `return_lag_7` | `daily_return` | `daily_return.shift(7)` | `features.py` | yes | Return seven trading days ago. |
| `close_lag_14` | `close` | `close.shift(14)` | `features.py` | yes | Close fourteen trading days ago. |
| `return_lag_14` | `daily_return` | `daily_return.shift(14)` | `features.py` | yes | Return fourteen trading days ago. |
| `vn30_return` | VN30 `close` | `VN30.close.pct_change()` | `features.py` / `cleaner.py` | yes | Broad market index return context. |
| `vn30_close` | VN30 `close` | Raw VN30 close joined by date | `features.py` / `cleaner.py` | yes | Broad market index level. |
| `vn30f_return` | VN30F1M `close` | `VN30F1M.close.pct_change()` | `features.py` / `cleaner.py` | yes | Futures market return context. |
| `vn30f_close` | VN30F1M `close` | Raw VN30F1M close joined by date | `features.py` / `cleaner.py` | yes | Futures market level. |

Current implementation note: there are no direct cross-stock features such as VIC/VHM spread or Vingroup basket return. The only cross-market features are VN30 and VN30F1M context features.

## Target Inventory

| Question | Current implementation |
|---|---|
| Does the model predict price or return? | The model trains on future return, then converts predicted return back to price. |
| Target formula | `target = (close.shift(-step) - close) / close` |
| Horizon | Direct multi-step horizons `step=1` through `step=7`. |
| Where target is created | `QuantileLightGBM.prepare_data(df, step)` and `validation._prepare_supervised_data(...)`. |
| Is target persisted to SQLite? | No. It is transient during training/validation. |
| Do `future_return_1` to `future_return_7` columns exist? | No. No persistent `future_return_*` columns were found. |
| How is price forecast reconstructed? | `pred_price = current_close * (1 + pred_return)` |
| Does validation use price or return metrics? | Model predicts return; validation reconstructs price and computes metrics on price forecast vs actual future price. |

## Metric Inventory

| Metric name | Where computed | Formula / meaning | Used for what | Easy explanation | Type |
|---|---|---|---|---|---|
| MAE | `trainer.evaluate_holdout`, `validation._calculate_metrics` | Mean absolute error between actual and predicted prices | Holdout and walk-forward model quality | Average absolute price miss | Model metric |
| RMSE | `trainer.evaluate_holdout`, `validation._calculate_metrics` | Square root of mean squared error | Holdout, walk-forward, governance RMSE gate | Penalizes large misses more than MAE | Model metric |
| MAPE | `trainer.evaluate_holdout`, `validation._calculate_metrics` | Mean absolute percentage error | Model status and governance primary metric | Average percentage miss | Model metric |
| SMAPE | `validation._calculate_metrics` | Symmetric percentage error using actual and predicted magnitude | Walk-forward validation report | Percent error that is less sensitive to scale | Model metric |
| Directional Accuracy | `validation._calculate_metrics` | Share of times predicted direction matches actual direction | Walk-forward validation and governance | Did the model call up/down correctly? | Model metric |
| Interval Coverage 80% | `validation._calculate_metrics` | Share of actual prices inside `[q_0.1, q_0.9]` | Walk-forward calibration check | How often the 80% band contains actual price | Model metric |
| Interval Coverage 95% | `validation._calculate_metrics` | Share of actual prices inside `[q_0.025, q_0.975]` | Walk-forward calibration and governance | How often the 95% band contains actual price | Model metric |
| Interval Coverage | `validation._calculate_metrics` | Alias currently set to 95% coverage | Report compatibility | General interval coverage field | Model metric |
| Pinball Loss | `validation._calculate_metrics` | Quantile loss averaged across configured quantiles | Quantile forecast quality | Penalizes quantile forecasts differently above/below actual | Model metric |
| Prediction Bias | `validation._calculate_metrics` | Mean predicted price minus actual price | Walk-forward report | Positive means overprediction, negative means underprediction | Model metric |
| Prediction Bias % | `validation._calculate_metrics` | Mean relative prediction error | Walk-forward report | Percent bias | Model metric |
| Quantile Crossing Rate | `validation._predict_quantile_prices`, `_calculate_metrics` | Share of rows with unordered raw quantile predictions | Validation diagnostics | How often quantiles came out in the wrong order before sorting | Model metric |
| Expected Return 7d | `risk_engine.calculate_risk_report` | `(T+7 q_0.5 - current_price) / current_price` | Risk report and signal logic | Median 7-day expected return | Risk metric |
| Downside Risk 95% | `risk_engine.calculate_risk_report` | `min(0, (T+7 q_0.025 - current_price) / current_price)` | Risk report | Lower-tail loss estimate | Risk metric |
| Upside Potential 95% | `risk_engine.calculate_risk_report` | `max(0, (T+7 q_0.975 - current_price) / current_price)` | Risk report | Upper-tail upside estimate | Risk metric |
| VaR 95% | `risk_engine.calculate_risk_report` | `max(0, (current_price - q_0.025) / current_price)` | Risk level and signal logic | Estimated 95% downside loss magnitude | Risk metric |
| Expected Shortfall | `risk_engine.calculate_risk_report` | `VaR_95 * 1.15` simple approximation | Risk level | Rough average loss beyond VaR | Risk metric |
| Risk/Reward Ratio | `risk_engine.calculate_risk_report` | `upside_potential_95 / abs(downside_risk_95)` | BUY/SELL/WATCH logic | Upside compared with downside | Risk metric |
| Risk Level | `risk_engine._risk_level` | LOW/MEDIUM/HIGH/EXTREME label from VaR, ES, drift, regime | Risk report and signal cap | Overall risk bucket | Risk metric |
| Signal Confidence | `risk_engine._signal_confidence` | Score adjusted by directional accuracy, coverage, risk, drift | Final signal confidence | Confidence in research signal | Risk metric |
| Drift Severity | `drift_detector._severity` | LOW/MEDIUM/HIGH label from feature/target/concept drift | Drift report and risk level | How serious current drift is | Drift metric |
| PSI | `drift_detector._psi` | Population Stability Index for selected features | Feature drift detection | Distribution shift score | Drift metric |
| Feature Mean Shift Z | `drift_detector.detect_drift` | `abs(current_mean - reference_mean) / reference_std` | Feature drift detection | Mean moved by how many reference standard deviations | Drift metric |
| Feature Std Ratio | `drift_detector.detect_drift` | `current_std / reference_std` | Feature drift detection | Volatility/spread shift of feature distribution | Drift metric |
| Target Return Mean Shift | `drift_detector._target_drift` | Mean shift z-score on close returns | Target drift detection | Return distribution shifted | Drift metric |
| Target Volatility Ratio | `drift_detector._target_drift` | Current return std / reference return std | Target drift detection | Current target volatility changed | Drift metric |
| Regime Confidence | `regime_detector._confidence` | Heuristic score from row count and regime clarity | Regime report | Confidence in regime label | Regime metric |
| Volatility Percentile | `regime_detector.detect_regime` | Percentile rank of current 20-day volatility | Volatility regime | Current vol relative to history | Regime metric |
| Sharpe Ratio | Not found | Not implemented | Not used | Risk-adjusted return metric | Not implemented |
| Max Drawdown | Not found | Not implemented | Not used | Worst peak-to-trough loss | Not implemented |
| Win Rate | Not found | Not implemented | Not used | Share of winning trades/signals | Not implemented |

## Agent Inventory

### AgentState Fields

`AgentState` currently defines these optional fields:

```text
ticker
run_id
forecast_data
quantiles_fixed
evaluation_status
evaluation_reason
validation_metrics
regime_report
drift_report
risk_report
news_context
news_found
news_items_count
news_items
evidence_level
shock_type
news_analysis
news_evidence
trading_signal
signal_confidence
governance_decision
agent_assessment_summary
retry_count
action_taken
final_recommendation
audit_trail
```

### LangGraph Nodes

| Node | Function | What it does |
|---|---|---|
| `node_validate` | `node_validate` | Sorts quantile outputs if crossing is detected. |
| `node_evaluate` | `node_evaluate` | Compares holdout MAPE with `thresholds.max_mape`. |
| `node_contextualize` | `node_contextualize` | Searches RSS news and classifies shock context. |
| `node_improve` | `node_improve` | Adjusts model config, retrains challenger, accepts or rolls back. |
| `node_recommend` | `node_recommend` | Uses risk report for final research signal and Gemini for committee-style summary. |

### Router Logic

| Router | Logic |
|---|---|
| `route_after_evaluate` | PASS goes to recommend; first ABNORMAL goes to news contextualization; retry ABNORMAL goes to improve. |
| `route_after_contextualize` | BLACK_SWAN/DATA_ISSUE goes to recommend; otherwise goes to improve. |
| `route_after_improve` | If retry count reaches max, goes to recommend; otherwise validates and evaluates again. |

### Agent Permissions and Behavior

| Question | Answer |
|---|---|
| Does the agent use Gemini? | Yes, in `src/agent/nodes.py`. |
| Where are prompts? | Inline in `_invoke_context_committee` and `_invoke_recommendation_committee`. |
| Can the agent modify config? | Yes. `tool_adjust_model_hyperparams` writes `configs/model_config.yaml`. |
| Can the agent retrain? | Yes. `node_improve` calls `generate_7_day_forecast` after config adjustment. |
| Can the agent rollback? | Yes. If challenger fails governance gates, backup config is written back and champion forecast is kept. |
| Can the agent decide signal? | The final `trading_signal` is assigned from `risk_report["preliminary_signal"]`; Gemini summarizes evidence rather than freely deciding trades. |
| Does the system place orders? | No. No broker API or order execution exists. |

### Shock Types

Allowed shock types:

```text
NO_NEWS
TREND_SHIFT
EVENT_DRIVEN
BLACK_SWAN
DATA_ISSUE
MODEL_DEGRADATION
```

## Output Inventory

### SQLite Tables

Current observed table patterns:

| Table | Contents |
|---|---|
| `raw_VIC` | Raw VIC OHLCV from vnstock. |
| `raw_VHM` | Raw VHM OHLCV from vnstock. |
| `raw_VRE` | Raw VRE OHLCV from vnstock. |
| `raw_VPL` | Raw VPL OHLCV from vnstock. |
| `raw_VN30` | Raw VN30 OHLCV context. |
| `raw_VN30F1M` | Raw VN30F1M OHLCV context. |
| `processed_VIC` | VIC raw columns plus engineered features and context columns. |
| `processed_VHM` | VHM processed table. |
| `processed_VRE` | VRE processed table. |
| `processed_VPL` | VPL processed table. |

### Reports

| Output | Location |
|---|---|
| JSON report | `reports/json/{ticker}_report_{YYYY-MM-DD}.json` |
| Markdown report | `reports/markdown/{ticker}_report_{YYYY-MM-DD}.md` |
| HTML report | `reports/html/{ticker}_report_{YYYY-MM-DD}.html` |

### Logs

| Output | Location |
|---|---|
| Runtime log | `logs/{YYYY-MM-DD}_system.log` |
| Console log | Standard terminal output via `logging.StreamHandler` |

### Model Artifacts

No persisted model artifact file is currently written. LightGBM models are trained in memory during forecast and validation.

### Config Changes

`src/agent/tools.py` may mutate `configs/model_config.yaml` during retrain experiments. If a challenger is rejected, `node_improve` restores the backed-up config. If a challenger is accepted, the adjusted config remains in place.

Current implementation note: because model config can be changed during runtime, reviewers should inspect `configs/model_config.yaml` before a final assessment run.


# Metrics Guide

Tài liệu này giải thích các metric quan trọng trong project bằng tiếng Việt, theo hướng dễ hiểu cho người mới học quantitative research / machine learning.

Lưu ý: một số metric trong danh sách assessment chưa được implement trong code hiện tại. Những metric đó được ghi rõ là "chưa implement".

## 1. MAE

### Tên metric

Mean Absolute Error.

### Công thức đơn giản

```text
MAE = trung bình(|giá thực tế - giá dự báo|)
```

### Ý nghĩa

MAE đo trung bình model dự báo sai bao nhiêu đơn vị giá.

Ví dụ nếu MAE = 5,000 VND, nghĩa là trung bình forecast lệch khoảng 5,000 VND so với giá thực tế.

### Dùng ở đâu trong project

- `src/modeling/trainer.py`: holdout evaluation.
- `src/modeling/validation.py`: walk-forward validation.

### Ví dụ dễ hiểu

Giá thực tế là 100, 110, 120. Model dự báo 98, 115, 118.

Sai số tuyệt đối là 2, 5, 2. MAE = 3.

### Khi nào metric báo động

MAE cao so với mức giá cổ phiếu nghĩa là model đang lệch nhiều. Tuy nhiên MAE phụ thuộc scale giá, nên nên đọc cùng MAPE.

## 2. RMSE

### Tên metric

Root Mean Squared Error.

### Công thức đơn giản

```text
RMSE = sqrt(trung bình((giá thực tế - giá dự báo)^2))
```

### Ý nghĩa

RMSE giống MAE nhưng phạt lỗi lớn mạnh hơn.

### Dùng ở đâu trong project

- `src/modeling/trainer.py`: holdout evaluation.
- `src/modeling/validation.py`: walk-forward validation.
- `src/agent/nodes.py`: governance gate, không cho challenger tệ hơn quá nhiều về RMSE.

### Ví dụ dễ hiểu

Nếu model sai nhỏ đều đều thì RMSE gần MAE. Nếu có vài ngày sai cực lớn, RMSE sẽ tăng mạnh.

### Khi nào metric báo động

RMSE tăng nhanh so với MAE thường cho thấy model có các lỗi lớn bất thường.

## 3. MAPE

### Tên metric

Mean Absolute Percentage Error.

### Công thức đơn giản

```text
MAPE = trung bình(|giá thực tế - giá dự báo| / |giá thực tế|)
```

### Ý nghĩa

MAPE đo lỗi theo phần trăm, dễ so sánh hơn MAE khi giá cổ phiếu có scale khác nhau.

### Dùng ở đâu trong project

- `src/modeling/trainer.py`: holdout evaluation.
- `src/modeling/validation.py`: walk-forward validation.
- `src/agent/nodes.py`: `node_evaluate` so sánh holdout MAPE với `configs/agent_config.yaml`.
- `src/agent/nodes.py`: governance gate yêu cầu challenger phải cải thiện MAPE.

### Ví dụ dễ hiểu

Giá thực tế 100, dự báo 95. Lỗi là 5%. MAPE càng thấp càng tốt.

### Khi nào metric báo động

Khi MAPE vượt threshold trong `agent_config.yaml`, agent đánh dấu model là `ABNORMAL` và có thể kích hoạt retrain loop.

Current implementation note: file config hiện tại là source of truth. Nếu assessment cần demo threshold 1%, hãy kiểm tra lại `configs/agent_config.yaml` trước khi chạy.

## 4. SMAPE

### Tên metric

Symmetric Mean Absolute Percentage Error.

### Công thức đơn giản

```text
SMAPE = trung bình(2 * |actual - predicted| / (|actual| + |predicted|))
```

### Ý nghĩa

SMAPE là phiên bản phần trăm cân bằng hơn giữa giá thực tế và giá dự báo.

### Dùng ở đâu trong project

- `src/modeling/validation.py`: walk-forward validation.

### Ví dụ dễ hiểu

Nếu actual = 100 và predicted = 110, SMAPE dùng cả 100 và 110 ở mẫu số, thay vì chỉ dùng actual như MAPE.

### Khi nào metric báo động

SMAPE cao nghĩa là forecast kém ổn định theo phần trăm. Nên đọc cùng MAPE.

## 5. Directional Accuracy

### Tên metric

Directional Accuracy.

### Công thức đơn giản

```text
Directional Accuracy =
  tỷ lệ ngày model dự báo đúng chiều tăng/giảm
```

Trong code:

```text
sign(actual_price - current_price) == sign(predicted_price - current_price)
```

### Ý nghĩa

Metric này không hỏi "dự báo đúng bao nhiêu VND", mà hỏi "dự báo đúng hướng không?".

### Dùng ở đâu trong project

- `src/modeling/validation.py`: walk-forward validation.
- `src/agent/nodes.py`: governance gate.
- `src/risk/risk_engine.py`: signal confidence và signal cap.

### Ví dụ dễ hiểu

Nếu cổ phiếu thực tế tăng và model cũng dự báo tăng, đó là đúng hướng.

### Khi nào metric báo động

Nếu directional accuracy dưới khoảng 50%, model đang yếu trong việc gọi đúng chiều. Risk engine có thể hạ signal về `WATCH` hoặc `HOLD`.

## 6. Interval Coverage

### Tên metric

Interval Coverage 80% và Interval Coverage 95%.

### Công thức đơn giản

```text
Coverage = tỷ lệ giá thực tế nằm trong khoảng dự báo
```

80% interval:

```text
q_0.1 <= actual <= q_0.9
```

95% interval:

```text
q_0.025 <= actual <= q_0.975
```

### Ý nghĩa

Coverage đo xem khoảng dự báo có "ôm" được giá thực tế hay không.

### Dùng ở đâu trong project

- `src/modeling/validation.py`: walk-forward validation.
- `src/agent/nodes.py`: governance gate dùng 95% interval coverage.
- `src/risk/risk_engine.py`: signal confidence.

### Ví dụ dễ hiểu

Nếu model nói khoảng 95% là 90-110, và giá thực tế là 105, lần đó được tính là covered.

### Khi nào metric báo động

Coverage quá thấp nghĩa là interval quá tự tin hoặc model không calibrated. Coverage cao nhưng interval quá rộng cũng có thể không hữu ích, nhưng project hiện chưa đo interval width riêng.

## 7. Pinball Loss

### Tên metric

Pinball Loss / Quantile Loss.

### Công thức đơn giản

Với quantile `q`:

```text
loss = max(q * residual, (q - 1) * residual)
residual = actual - predicted_quantile
```

### Ý nghĩa

Pinball loss là metric chuẩn cho quantile regression. Nó đánh giá chất lượng từng quantile, không chỉ median.

### Dùng ở đâu trong project

- `src/modeling/validation.py`: walk-forward validation.

### Ví dụ dễ hiểu

Với quantile thấp như 2.5%, model không bị phạt giống nhau khi dự báo trên hoặc dưới actual. Cách phạt bất đối xứng này giúp model học đúng vị trí quantile.

### Khi nào metric báo động

Pinball loss tăng nghĩa là các quantile forecast đang kém hơn. Nên so sánh theo thời gian hoặc giữa champion/challenger.

## 8. Prediction Bias

### Tên metric

Prediction Bias.

### Công thức đơn giản

```text
Prediction Bias = trung bình(predicted_price - actual_price)
```

### Ý nghĩa

Bias cho biết model có xu hướng dự báo cao hơn hay thấp hơn thực tế.

### Dùng ở đâu trong project

- `src/modeling/validation.py`: walk-forward validation.
- Report Markdown/JSON.

### Ví dụ dễ hiểu

Nếu bias = -500, model thường dự báo thấp hơn thực tế khoảng 500 VND.

### Khi nào metric báo động

Bias lớn theo một chiều cho thấy model có lỗi hệ thống, không chỉ nhiễu ngẫu nhiên.

## 9. VaR

### Tên metric

Value at Risk 95%.

### Công thức đơn giản trong project

```text
VaR_95 = max(0, (current_price - q_0.025_T+7) / current_price)
```

### Ý nghĩa

VaR 95% ước lượng mức lỗ downside ở vùng tail 5% theo forecast quantile.

### Dùng ở đâu trong project

- `src/risk/risk_engine.py`: risk report và signal logic.

### Ví dụ dễ hiểu

Nếu current price = 100 và q_0.025 = 92, VaR 95% = 8%.

### Khi nào metric báo động

VaR cao cho thấy downside tail risk lớn. Trong code, VaR cao có thể nâng `risk_level` lên `MEDIUM` hoặc `HIGH`.

## 10. Expected Shortfall

### Tên metric

Expected Shortfall.

### Công thức đơn giản trong project

```text
Expected Shortfall = VaR_95 * 1.15
```

### Ý nghĩa

Expected Shortfall cố gắng mô tả mức lỗ trung bình khi tình huống xấu hơn VaR xảy ra. Trong project hiện tại, đây là approximation đơn giản, không phải expected shortfall tính từ full distribution.

### Dùng ở đâu trong project

- `src/risk/risk_engine.py`: risk report và risk level.

### Ví dụ dễ hiểu

Nếu VaR 95% = 10%, expected shortfall trong project = 11.5%.

### Khi nào metric báo động

Expected shortfall cao nghĩa là tail risk nghiêm trọng. Code có thể phân loại risk là `HIGH`.

## 11. Risk/Reward Ratio

### Tên metric

Risk/Reward Ratio.

### Công thức đơn giản trong project

```text
risk_reward_ratio = upside_potential_95 / abs(downside_risk_95)
```

### Ý nghĩa

Metric này so sánh upside tiềm năng với downside risk.

### Dùng ở đâu trong project

- `src/risk/risk_engine.py`: quyết định preliminary signal.

### Ví dụ dễ hiểu

Nếu upside = 12% và downside = 6%, risk/reward = 2.0.

### Khi nào metric báo động

Risk/reward thấp nghĩa là upside không đủ hấp dẫn so với downside.

## 12. Sharpe Ratio

### Tên metric

Sharpe Ratio.

### Công thức đơn giản

```text
Sharpe = (portfolio_return - risk_free_rate) / return_volatility
```

### Ý nghĩa

Sharpe đo lợi nhuận điều chỉnh theo rủi ro.

### Dùng ở đâu trong project

Chưa implement. Không tìm thấy module backtest hoặc portfolio return dùng Sharpe ratio.

### Ví dụ dễ hiểu

Hai chiến lược có return 10%, nhưng chiến lược biến động thấp hơn sẽ có Sharpe tốt hơn.

### Khi nào metric báo động

Sharpe thấp hoặc âm báo hiệu chiến lược không tạo đủ return so với rủi ro.

## 13. Max Drawdown

### Tên metric

Maximum Drawdown.

### Công thức đơn giản

```text
Max Drawdown = mức giảm lớn nhất từ đỉnh vốn xuống đáy vốn
```

### Ý nghĩa

Max drawdown đo khoản lỗ tệ nhất nếu đi từ peak đến trough.

### Dùng ở đâu trong project

Chưa implement. Project hiện chưa có portfolio-level backtest.

### Ví dụ dễ hiểu

Nếu equity curve tăng lên 100 rồi giảm xuống 70 trước khi hồi phục, drawdown là 30%.

### Khi nào metric báo động

Drawdown lớn cho thấy chiến lược có rủi ro chịu lỗ sâu.

## 14. Win Rate

### Tên metric

Win Rate.

### Công thức đơn giản

```text
Win Rate = số giao dịch thắng / tổng số giao dịch
```

### Ý nghĩa

Win rate đo tỷ lệ signal/trade có kết quả dương.

### Dùng ở đâu trong project

Chưa implement. Project hiện chỉ output research signal, không có trade execution hoặc trade ledger.

### Ví dụ dễ hiểu

Nếu 60 trên 100 trade lời, win rate = 60%.

### Khi nào metric báo động

Win rate thấp có thể xấu, nhưng phải đọc cùng payoff ratio. Một chiến lược win rate thấp vẫn có thể tốt nếu lãi mỗi lần thắng lớn hơn lỗ mỗi lần thua.

## 15. Drift Severity

### Tên metric

Drift Severity.

### Công thức / logic

Project gán nhãn:

```text
LOW / MEDIUM / HIGH
```

dựa trên:

- feature drift
- target drift
- concept drift
- số feature bị drift

### Dùng ở đâu trong project

- `src/monitoring/drift_detector.py`
- `src/risk/risk_engine.py`
- report JSON/Markdown/HTML

### Ví dụ dễ hiểu

Nếu nhiều feature hiện tại khác mạnh so với lịch sử, severity có thể là `HIGH`.

### Khi nào metric báo động

`HIGH` thường dẫn tới `MANUAL_REVIEW` trong risk engine.

## 16. Regime Confidence

### Tên metric

Regime Confidence.

### Công thức / logic

Heuristic score dựa trên:

- số dòng dữ liệu
- volatility regime có rõ không
- trend regime có rõ không
- liquidity regime có bất thường không

### Dùng ở đâu trong project

- `src/monitoring/regime_detector.py`
- report output

### Ví dụ dễ hiểu

Nếu dữ liệu đủ dài và trend/volatility rõ, confidence cao hơn.

### Khi nào metric báo động

Confidence thấp nghĩa là regime label nên được đọc thận trọng.

## 17. Signal Confidence

### Tên metric

Signal Confidence.

### Công thức / logic

Risk engine bắt đầu từ base confidence rồi điều chỉnh theo:

- directional accuracy
- interval coverage
- risk level
- drift severity
- loại signal

### Dùng ở đâu trong project

- `src/risk/risk_engine.py`
- `AgentState`
- final report

### Ví dụ dễ hiểu

Nếu model có directional accuracy tốt và coverage tốt, confidence tăng. Nếu drift cao hoặc risk extreme, confidence giảm.

### Khi nào metric báo động

Confidence thấp nghĩa là signal nên được xem là yếu hoặc cần manual review.


# Pipeline Walkthrough VI

Tài liệu này giải thích toàn bộ pipeline bằng tiếng Việt, theo kiểu hướng dẫn lại cho người mới đọc project.

## 1. Dự án này làm gì?

Project **Agentic Vingroup Quant Forecasting System** là một hệ thống research pipeline cho cổ phiếu nhóm Vingroup.

Nó làm các việc chính:

1. Lấy dữ liệu giá/khối lượng từ vnstock.
2. Lưu dữ liệu vào SQLite.
3. Tạo feature kỹ thuật.
4. Train model LightGBM Quantile.
5. Forecast 7 ngày tiếp theo.
6. Đánh giá model bằng holdout và walk-forward validation.
7. Kiểm tra market regime.
8. Kiểm tra drift.
9. Tính risk report.
10. Cho LangGraph Agent đánh giá model và governance.
11. Nếu cần, thử retrain model challenger.
12. Nếu challenger tốt hơn theo governance gate thì accept, nếu không thì rollback/giữ champion.
13. Xuất report JSON, Markdown, HTML.

Điểm quan trọng: hệ thống này chỉ tạo **research signal**, không đặt lệnh thật.

## 2. Dữ liệu đi từ đâu tới đâu?

### Bước 1: Lấy dữ liệu

File:

```text
src/ingestion/vnstock_api.py
```

Hàm chính:

```python
get_vingroup_and_context_data(start_date, end_date)
```

Hàm này lấy:

- `VIC`
- `VHM`
- `VRE`
- `VPL`
- `VN30`
- `VN30F1M`

Dữ liệu là daily OHLCV:

```text
open, high, low, close, volume, ticker
```

### Bước 2: Lưu raw data

File:

```text
src/processing/cleaner.py
src/processing/db_manager.py
```

Raw data được lưu vào SQLite:

```text
data/database.sqlite
```

Các bảng raw:

```text
raw_VIC
raw_VHM
raw_VRE
raw_VPL
raw_VN30
raw_VN30F1M
```

### Bước 3: Tạo feature

File:

```text
src/processing/features.py
```

Hàm:

```python
generate_technical_features(df)
generate_context_features(context_df, prefix)
```

Feature tạo ra gồm:

- daily return
- volume change
- moving average
- volatility
- RSI
- MACD
- ATR
- ROC
- calendar features
- lag features
- VN30 context
- VN30F1M context

### Bước 4: Lưu processed data

Processed data được lưu vào:

```text
processed_VIC
processed_VHM
processed_VRE
processed_VPL
```

Đây là dữ liệu chính model sử dụng.

## 3. Model học cái gì?

Model nằm ở:

```text
src/modeling/trainer.py
src/modeling/predictor.py
```

Class chính:

```python
QuantileLightGBM
```

Hàm quan trọng:

```python
prepare_data(df, step)
```

Model không học trực tiếp raw price tương lai. Nó học **future return**:

```text
target = (close[t + step] - close[t]) / close[t]
```

Ví dụ:

- hôm nay close = 100
- 7 ngày sau close = 110
- target return = 10%

Khi model predict xong return, project convert lại thành price:

```text
predicted_price = current_close * (1 + predicted_return)
```

## 4. Forecast 7 ngày hoạt động ra sao?

File:

```text
src/modeling/predictor.py
```

Hàm:

```python
generate_7_day_forecast(df)
```

Pipeline forecast:

1. Tạo trainer `QuantileLightGBM`.
2. Chạy holdout evaluation.
3. Chạy walk-forward validation.
4. Lấy dòng dữ liệu mới nhất làm input.
5. Lặp `step=1` tới `step=7`.
6. Với mỗi step, train nhiều model quantile.
7. Trả ra các forecast:

```text
q_0.025
q_0.1
q_0.5
q_0.9
q_0.975
```

Ý nghĩa:

- `q_0.5`: median forecast
- `q_0.1` và `q_0.9`: khoảng 80%
- `q_0.025` và `q_0.975`: khoảng 95%

## 5. Agent làm gì?

Agent nằm trong:

```text
src/agent/
```

Các file chính:

```text
state.py
graph.py
nodes.py
tools.py
```

Agent không phải trader tự động. Nó đóng vai trò:

- Quantitative Risk Committee Agent
- Model Governance Reviewer
- Market Context Analyst

Agent đọc:

- forecast data
- holdout metrics
- walk-forward metrics
- regime report
- drift report
- risk report
- news evidence
- governance decision

Agent output:

- model status
- shock type
- news assessment
- governance decision
- final research signal
- final recommendation text
- audit trail

## 6. Agent workflow chạy như thế nào?

File:

```text
src/agent/graph.py
src/agent/nodes.py
```

Graph gồm 5 node:

```text
node_validate
node_evaluate
node_contextualize
node_improve
node_recommend
```

Luồng cơ bản:

```text
node_validate
  -> node_evaluate
      -> nếu PASS: node_recommend
      -> nếu ABNORMAL lần đầu: node_contextualize
      -> nếu ABNORMAL sau retry: node_improve
```

Sau contextualize:

```text
BLACK_SWAN hoặc DATA_ISSUE -> node_recommend
các trường hợp khác -> node_improve
```

Sau improve:

```text
nếu còn retry -> quay lại node_validate
nếu hết retry -> node_recommend
```

## 7. Vì sao cần retrain?

Model có thể xuống chất lượng vì:

- thị trường đổi regime
- distribution của feature thay đổi
- volatility tăng
- dữ liệu gần đây khác dữ liệu lịch sử
- model hiện tại underfit/overfit

Trong project, retrain được kích hoạt khi:

```text
holdout MAPE > thresholds.max_mape
```

Threshold nằm ở:

```text
configs/agent_config.yaml
```

Current implementation note: file config hiện tại là source of truth. Nếu bài assessment yêu cầu demo threshold 1%, hãy kiểm tra file này trước khi chạy.

## 8. Vì sao có rollback?

Không phải model retrain là tốt hơn.

Một challenger có thể:

- MAPE tốt hơn một chút
- nhưng directional accuracy tệ hơn
- hoặc interval coverage tệ hơn
- hoặc RMSE tệ hơn

Vì vậy project dùng governance gate:

```text
MAPE phải cải thiện
Directional accuracy không được giảm quá nhiều
95% interval coverage không được giảm quá nhiều
RMSE không được xấu hơn quá nhiều
```

Nếu challenger fail, project rollback config và giữ champion.

File:

```text
src/agent/nodes.py
```

Hàm:

```python
_compare_champion_challenger(...)
```

## 9. Regime detection là gì?

File:

```text
src/monitoring/regime_detector.py
```

Hàm:

```python
detect_regime(processed_df)
```

Regime detector trả ra:

```text
volatility_regime
trend_regime
liquidity_regime
regime_confidence
regime_notes
```

Ví dụ:

```text
NORMAL_VOLATILITY
UPTREND
NORMAL_LIQUIDITY
```

Regime giúp risk engine hiểu forecast đang nằm trong môi trường thị trường nào.

## 10. Drift detection là gì?

File:

```text
src/monitoring/drift_detector.py
```

Hàm:

```python
detect_drift(reference_df, current_df, validation_metrics)
```

Drift gồm 3 loại:

### Feature drift

Feature hiện tại khác mạnh so với feature lịch sử.

Ví dụ:

- volume tăng bất thường
- RSI distribution khác trước
- volatility feature khác trước

### Target drift

Return của giá close thay đổi distribution.

### Concept drift

Quan hệ giữa feature và target có thể đã thay đổi. Project suy luận concept drift từ metric validation:

- MAPE quá cao
- directional accuracy yếu
- interval coverage thấp

Nếu drift severity là `HIGH`, risk engine có thể đưa signal về `MANUAL_REVIEW`.

## 11. Risk engine làm gì?

File:

```text
src/risk/risk_engine.py
```

Hàm:

```python
calculate_risk_report(...)
```

Risk engine đọc forecast quantiles và tính:

- expected return 7 ngày
- downside risk 95%
- upside potential 95%
- VaR 95%
- expected shortfall
- risk/reward ratio
- risk level
- preliminary signal
- signal confidence

Signal có thể là:

```text
BUY
SELL
HOLD
WATCH
MANUAL_REVIEW
```

Nếu drift cao hoặc volatility extreme, signal thường bị đưa về `MANUAL_REVIEW`.

## 12. Báo cáo output đọc như thế nào?

Reports nằm ở:

```text
reports/json/
reports/markdown/
reports/html/
```

### JSON report

Dành cho máy đọc hoặc debug sâu. Nó chứa gần như toàn bộ `AgentState`.

Nên xem JSON khi bạn muốn biết:

- forecast data đầy đủ
- risk_report
- drift_report
- regime_report
- governance_decision
- audit_trail

### Markdown report

Dành cho người đọc. Có cấu trúc:

1. Executive Summary
2. Forecast Performance
3. Walk-forward Validation
4. Market Regime
5. Drift Detection
6. Risk Assessment
7. News & Event Context
8. Model Governance Decision
9. Final Research Signal
10. Audit Trail

### HTML report

Dành cho xem nhanh trong browser. Có summary panel và forecast fan chart bằng Plotly.

## 13. Nếu tôi muốn debug thì xem file nào trước?

### Pipeline không chạy

Xem:

```text
main.py
logs/YYYY-MM-DD_system.log
```

### Không lấy được data

Xem:

```text
src/ingestion/vnstock_api.py
```

Kiểm tra:

- network
- vnstock API
- symbol
- date range

### Data đã lấy nhưng feature lỗi

Xem:

```text
src/processing/features.py
src/processing/cleaner.py
data/database.sqlite
```

### Model forecast lỗi

Xem:

```text
src/modeling/trainer.py
src/modeling/predictor.py
src/modeling/validation.py
```

### Drift/regime/risk lạ

Xem:

```text
src/monitoring/regime_detector.py
src/monitoring/drift_detector.py
src/risk/risk_engine.py
```

### Agent routing lạ

Xem:

```text
src/agent/graph.py
src/agent/nodes.py
src/agent/state.py
```

### News luôn NO_NEWS

Xem:

```text
src/agent/tools.py
```

Có thể RSS parse lỗi hoặc keyword không match.

### Report thiếu field

Xem:

```text
src/reporting/generator.py
reports/json/
```

JSON report là nơi tốt nhất để kiểm tra field thực tế.

## 14. File đọc theo thứ tự đề xuất

Nếu bạn mới quay lại project sau một thời gian, nên đọc theo thứ tự:

1. `README.md`
2. `docs/CODEBASE_REVIEW.md`
3. `main.py`
4. `src/processing/features.py`
5. `src/modeling/trainer.py`
6. `src/modeling/predictor.py`
7. `src/modeling/validation.py`
8. `src/monitoring/regime_detector.py`
9. `src/monitoring/drift_detector.py`
10. `src/risk/risk_engine.py`
11. `src/agent/graph.py`
12. `src/agent/nodes.py`
13. `src/reporting/generator.py`

## 15. Cách hiểu final signal

Final signal không phải lời khuyên đầu tư.

Ý nghĩa thực tế:

| Signal | Cách hiểu |
|---|---|
| `BUY` | Forecast/risk tương đối thuận lợi theo rule hiện tại. |
| `SELL` | Forecast/risk nghiêng tiêu cực. |
| `HOLD` | Không đủ edge rõ ràng để hành động. |
| `WATCH` | Có tín hiệu cần theo dõi nhưng chưa đủ mạnh. |
| `MANUAL_REVIEW` | Model/data/risk có vấn đề, cần con người kiểm tra. |

Trong môi trường assessment, `MANUAL_REVIEW` không phải lỗi. Nó có thể là output đúng nếu drift hoặc tail risk cao.
